# Geração de Tabela LaTeX a partir de arquivos .txt

Este notebook lê todos os arquivos `.txt` da pasta `input`, processa os dados e gera uma tabela em formato LaTeX para ser utilizada no relatório.

## 1. Importar bibliotecas necessárias
Vamos importar as bibliotecas pandas e os para manipulação de arquivos e dados.

In [13]:
import pandas as pd
import os
from pathlib import Path

## 2. Estrutura de pastas

In [14]:
from pathlib import Path
import shutil

# Caminhos absolutos (ajuste conforme necessário)
src_base = Path.cwd().parent / 'src' / 'dados'
dst_base = Path.cwd().parent / 'relatorioCefet' / 'tabelas'

# Remove a pasta de destino inteira se existir (limpeza total)
if dst_base.exists() and dst_base.is_dir():
    shutil.rmtree(dst_base)
    print(f'Pasta de destino removida: {dst_base}')

# Cria input e output novamente
(dst_base / 'input').mkdir(parents=True, exist_ok=True)
(dst_base / 'output').mkdir(parents=True, exist_ok=True)

# Cria as subpastas de output conforme existem em src/dados/output
output_src = src_base / 'output'
if output_src.exists():
    for subpasta in output_src.iterdir():
        if subpasta.is_dir():
            (dst_base / 'output' / subpasta.name).mkdir(parents=True, exist_ok=True)
print('Estrutura de pastas criada em:', dst_base)


Pasta de destino removida: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas
Estrutura de pastas criada em: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas


## 2. Listar arquivos .csv na pasta input
Vamos listar todos os arquivos `.csv` presentes na pasta `input`.

In [15]:
from pathlib import Path

# Define os caminhos das pastas
input_dir = Path('../src/dados/input').resolve()
output_dir = Path('../src/dados/output').resolve()

# Lista todos os arquivos em input (não recursivo)
arquivos_input = [arq for arq in input_dir.glob('*') if arq.is_file()]

# Lista todos os arquivos em output (recursivo, incluindo subpastas)
arquivos_output = [arq for arq in output_dir.rglob('*') if arq.is_file()]

print(f'Arquivos em input ({len(arquivos_input)}):')
for arq in arquivos_input:
    print(arq)

print(f'\nArquivos em output ({len(arquivos_output)}):')
for arq in arquivos_output:
    print(arq)

Arquivos em input (2):
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\input\entrada_parametros.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\input\entrada_tcl_parametros.csv

Arquivos em output (5):
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\output\saida_msd.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\output\saida_tcl_amostras.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\output\saida_tcl_resumo.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\output\saida_teste_hipotese.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\dados\output\saida_trajetorias.csv


## 3. Ler e processar arquivos .csv
Vamos ler o conteúdo de cada arquivo `.csv` e armazenar os dados em uma lista.

In [16]:
import pandas as pd
import ast
from pathlib import Path

# ============================================================
# Funções auxiliares
# ============================================================

def escapar_latex(texto):
    """Escapa caracteres especiais para LaTeX."""
    return (str(texto)
            .replace('\\', r'\textbackslash ')
            .replace('_', r'\_')
            .replace('%', r'\%')
            .replace('&', r'\&')
            .replace('#', r'\#')
            .replace('{', r'\{')
            .replace('}', r'\}')
            .replace('^', r'\^{}')
            .replace('~', r'\~{}')
           )


def gerar_caption(nome_base):
    nome = nome_base.replace('_', ' ').replace('-', ' ')
    nome = nome.capitalize()
    return f"Tabela – Dados referentes a {nome}."


def gerar_label(nome_base):
    nome = nome_base.replace('.', '_').replace('-', '_')
    return f"tab:{nome}"


# ============================================================
# Formatação automática de listas (dt_list e outras)
# ============================================================

def formatar_lista(valor):
    """
    Detecta automaticamente listas no CSV e formata para LaTeX.
    Funciona para listas longas, números ou strings.
    """
    try:
        lista = ast.literal_eval(valor)
        if not isinstance(lista, list):
            return escapar_latex(valor)
    except:
        return escapar_latex(valor)

    lista = [str(v) for v in lista]

    linhas = []
    for i in range(0, len(lista), 3):
        linhas.append(", ".join(lista[i:i+3]))

    conteudo = r" \\ ".join(linhas)

    return (
        r"\scriptsize{\begin{tabular}[c]{l}"
        + conteudo +
        r"\end{tabular}}"
    )


# ============================================================
# Função principal — gera tabela LaTeX
# ============================================================

def salvar_tabela_latex(arquivo_csv, pasta_saida):
    try:
        df = pd.read_csv(arquivo_csv)

        df_latex = df.copy()
        df_latex.columns = [escapar_latex(col) for col in df_latex.columns]

        # Formatação automática
        for col in df_latex.columns:
            df_latex[col] = df_latex[col].astype(str).apply(formatar_lista)

        nome_base = arquivo_csv.stem
        caption = gerar_caption(nome_base)
        label = gerar_label(nome_base)

        # Detecta colunas que são listas
        colunas_lista = []
        for col in df.columns:
            try:
                if df[col].astype(str).str.startswith("[").any():
                    colunas_lista.append(col)
            except:
                pass

        # Ajuste automático da largura da última coluna se houver listas
        if colunas_lista:
            col_format = " ".join(
                ["l"] * (len(df_latex.columns) - 1) + ["p{8cm}"]
            )
        else:
            col_format = None

        tabela = df_latex.to_latex(
            index=False,
            escape=False,
            column_format=col_format
        ) if col_format else df_latex.to_latex(index=False, escape=False)

        conteudo = (
            "\\begin{table}[H]\n"
            "\\centering\n"
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            f"{tabela}\n"
            "\\end{table}\n"
        )

        nome_saida = arquivo_csv.with_suffix(".tex").name
        caminho_saida = pasta_saida / nome_saida
        pasta_saida.mkdir(parents=True, exist_ok=True)

        with open(caminho_saida, "w", encoding="utf-8") as f:
            f.write(conteudo)

        print(f"Tabela LaTeX salva em: {caminho_saida}")

    except Exception as e:
        print(f"Erro ao processar {arquivo_csv}: {e}")


# ============================================================
# Processamento dos arquivos de INPUT
# ============================================================

pasta_saida_input = Path('../relatorioCefet/tabelas/input').resolve()

if not arquivos_input:
    print("Nenhum arquivo encontrado em input.")
else:
    for arquivo in arquivos_input:
        salvar_tabela_latex(arquivo, pasta_saida_input)


# ============================================================
# Processamento dos arquivos de OUTPUT (mantendo subpastas)
# ============================================================

pasta_saida_output = Path('../relatorioCefet/tabelas/output').resolve()
output_dir = Path('../src/dados/output').resolve()

if not arquivos_output:
    print("Nenhum arquivo encontrado em output.")
else:
    for arquivo in arquivos_output:
        subpath = arquivo.relative_to(output_dir).parent
        destino = pasta_saida_output / subpath
        destino.mkdir(parents=True, exist_ok=True)
        salvar_tabela_latex(arquivo, destino)

print("Processo finalizado.")

Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas\input\entrada_parametros.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas\input\entrada_tcl_parametros.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas\output\saida_msd.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas\output\saida_tcl_amostras.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas\output\saida_tcl_resumo.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\tabelas\output\saida_teste_hipotese.tex
Tabela LaTeX salv

## 8. Copiar toda a pasta de gráficos para figuras

Esta célula copia recursivamente todo o conteúdo da pasta `graficos` (incluindo subpastas e arquivos) para a pasta `relatorioCefet/figuras`. Útil para garantir que todos os gráficos estejam disponíveis no relatório.

In [17]:
import shutil
from pathlib import Path

# Caminhos de origem e destino
graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    # Remove a pasta de destino se já existir
    if destino.exists() and destino.is_dir():
        shutil.rmtree(destino)
        print(f'Pasta de destino removida: {destino}')
    # Cria a pasta de destino novamente
    destino.mkdir(parents=True, exist_ok=True)
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    for item in origem.rglob('*'):
        if item.is_file():
            destino_arquivo = destino / item.relative_to(origem)
            destino_arquivo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destino_arquivo)
            print(f'Arquivo copiado: {item} -> {destino_arquivo}')

copiar_pasta(graficos_dir, destino_graficos)


Pasta de destino removida: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\graficos\caminhadas.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos\caminhadas.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\graficos\msd_loglog.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos\msd_loglog.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\graficos\tcl.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos\tcl.png


In [18]:
import shutil
from pathlib import Path

# Caminhos de origem e destino
graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    for item in origem.rglob('*'):
        if item.is_file():
            destino_arquivo = destino / item.relative_to(origem)
            destino_arquivo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destino_arquivo)
            print(f'Arquivo copiado: {item} -> {destino_arquivo}')

copiar_pasta(graficos_dir, destino_graficos)



Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\graficos\caminhadas.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos\caminhadas.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\graficos\msd_loglog.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos\msd_loglog.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\src\graficos\tcl.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaGeradorRandonWalk\relatorioCefet\figuras\graficos\tcl.png


---
## Referências

- Slides `pmmat_aula07.pdf`: parâmetros do LCG e função `aleat()`.
- Slides `numerosaleatorios.pdf`: geradores congruenciais e aplicações.
- Random Walk 1D: distribuição binomial e aproximação gaussiana (TCL).